In [ ]:
import os
from PIL import Image
import torch
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt

from model import U2NET

In [ ]:
"""Normalize the predicted SOD probability map"""
def normPRED(d):
    ma = torch.max(d)
    mi = torch.min(d)
    dn = (d - mi) / (ma - mi)
    return dn

In [ ]:
def get_saliency_mask(image_path, model_path="/data_center/data2/dataset/chenwy/21164-data/model-ckpt/u2net/u2net.pth", model_name='u2net', input_size=320):
    # 1. PIL.Image -> convert to RGB
    image = Image.open(image_path).convert('RGB')
    original_shape = image.size  # (width, height)
    
    transform_pipeline = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    inputs_test = transform_pipeline(image).unsqueeze(0).to("cuda")
    
    print("...load U2NET---173.6 MB")
    net = U2NET(3, 1)
    net.load_state_dict(torch.load(model_path))
    net.cuda()
    net.eval()
    
    with torch.no_grad():
        d1, d2, d3, d4, d5, d6, d7 = net(inputs_test)
        
        pred = d1[:, 0, :, :]
        pred = normPRED(pred)
        pred_np = pred.squeeze().cpu().data.numpy()
        
        pred_pil = Image.fromarray((pred_np * 255).astype(np.uint8))
        pred_pil = pred_pil.resize(original_shape, resample=Image.BILINEAR)
        saliency_mask = np.array(pred_pil) / 255.0
    
    return saliency_mask


In [ ]:
def visualize_saliency(image_path, saliency_mask, figsize=(10, 5)):
    """
    可视化对比原始图像和 saliency mask
    
    Args:
        image_path: 原始图像路径
        saliency_mask: saliency mask (numpy数组，值范围 [0, 1])
        figsize: 图像显示大小
    """
    # 读取原始图像
    original_image = np.array(Image.open(image_path).convert('RGB'))
    
    # 创建对比图（2个子图）
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # 1. 原始图像
    axes[0].imshow(original_image)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')
    
    # 2. Saliency Mask (灰度图)
    axes[1].imshow(saliency_mask, cmap='gray')
    axes[1].set_title('Saliency Mask', fontsize=12)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    

In [ ]:
image_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/add_noise_denoise/chameleon_fake-random_add_noise_step/real/0a40e2a1-7b25-4575-b5f9-08401baaed32.png"

# 获取 saliency mask
mask = get_saliency_mask(image_path)

# 可视化对比
visualize_saliency(image_path, mask)
